In [6]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from datetime import datetime

In [17]:
# Config
IMG_SIZE = (50, 50)
BATCH_SIZE = 200
AUTOTUNE = tf.data.AUTOTUNE

# 1) Crear dataset con rutas + etiqueta
ds_normal = tf.data.Dataset.list_files('./imagenes/train/NORMAL/*.jpeg')
ds_normal = ds_normal.map(lambda p: (p, 0))

ds_pneumonia = tf.data.Dataset.list_files('./imagenes/train/PNEUMONIA/*.jpeg')
ds_pneumonia = ds_pneumonia.map(lambda p: (p, 1))

dataset = ds_normal.concatenate(ds_pneumonia)

ds_val_normal = tf.data.Dataset.list_files('./imagenes/val/NORMAL/*.jpeg')
ds_val_normal = ds_val_normal.map(lambda p: (p, 0))

ds_val_pneumonia = tf.data.Dataset.list_files('./imagenes/val/PNEUMONIA/*.jpeg')
ds_val_pneumonia = ds_val_pneumonia.map(lambda p: (p, 1))

# 2) Cargar y preparar imagen
def preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=1)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255
    return img, label

dataset = dataset.map(preprocess)
val_dataset = ds_val_normal.concatenate(ds_val_pneumonia).map(preprocess)

# 3) shuffle + batch + prefetch
dataset = dataset.shuffle(5216).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print(dataset.element_spec)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(50, 50, 1)),

    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.8),

    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.8),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.8),

    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.summary()

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                        patience=5)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2,   # Reduce by 80% (new_lr = lr * 0.2)
    patience=3,   # Wait 5 epochs
    min_lr=0.0001,
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath= 'checkpoint.model.keras',
    monitor='val_loss',
    verbose=0,
    save_best_only=True,
    save_weights_only=False,
    mode='auto',
    save_freq='epoch',
    initial_value_threshold=None
)

log_dir = "logs/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

model.fit(dataset, epochs=50, validation_data=val_dataset, callbacks=[stop, reduce_lr, checkpoint, tensorboard])


resultados = model.evaluate(dataset, return_dict=True)
print(resultados)

(TensorSpec(shape=(None, 50, 50, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 48, 48, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 22, 22, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 11, 11, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 11, 11, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 7744)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       991,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,010,305 (3.85 MB)

 Trainable params: 1,010,305 (3.85 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 7s 99ms/step - accuracy: 0.7111 - loss: 0.6770 - val_accuracy: 0.5000 - val_loss: 0.6931 - learning_rate: 0.0010
Epoch 2/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 6s 104ms/step - accuracy: 0.7431 - loss: 0.5792 - val_accuracy: 0.5000 - val_loss: 0.6917 - learning_rate: 0.0010
Epoch 3/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 6s 101ms/step - accuracy: 0.7429 - loss: 0.5523 - val_accuracy: 0.5000 - val_loss: 0.6892 - learning_rate: 0.0010
Epoch 4/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 6s 93ms/step - accuracy: 0.7429 - loss: 0.5085 - val_accuracy: 0.5000 - val_loss: 0.6853 - learning_rate: 0.0010
Epoch 5/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - accuracy: 0.7571 - loss: 0.4475 - val_accuracy: 0.5000 - val_loss: 0.6766 - learning_rate: 0.0010
Epoch 6/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 6s 97ms/step - accuracy: 0.7655 - loss: 0.4005 - val_accuracy: 0.6875 - val_loss: 0.6494 - learning_rate: 0.0010
Epoch 7/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 6s 97ms/step - accuracy: 0.7786 - loss: 0.3733 - val_a

In [8]:
pred = model.predict(dataset)

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
y_true = []
y_pred = []
for images, labels in dataset.unbatch():
    img = tf.expand_dims(images, axis=0)  # Añadir dimensión batch
    pred = model.predict(img)
    if pred[0][0] >= 0.5:
        y_pred.append(1)
    else:
        y_pred.append(0)
    y_true.append(int(labels.numpy()))

# 4. Calcular matriz de confusión
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Neumonía"])

fig, ax = plt.subplots()
disp.plot(
         cmap="Blues",                                   
         ax=ax,                                           
         colorbar=False                                   
         )
plt.show()

3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━

KeyboardInterrupt: 

In [18]:
%reload_ext tensorboard

In [19]:
%tensorboard --logdir ./logs/

Reusing TensorBoard on port 6008 (pid 17320), started 0:19:29 ago. (Use '!kill 17320' to kill it.)

##### Explicar como resuelve el problema la capa de salida.

La salida nos da el porcentaje de que la imagen tenga neumonía(clase 1) y el porcentaje restante es la probabilidad de que no la tenga(clase 0).

##### ¿Por qué el model actual tiene esa arquitectura y número de parametros?

Al ser un problema binario, lo mas importate es que la capa de salida tenga una neurona con activación sigmoide, para que nos de un resultado entre 0 y 1. El resto de la arquitectura es bastante común para problemas de clasificación de imágenes, con varias capas convolucionales seguidas de capas de pooling para reducir el tamaño.

##### Explicar que pasa con el val y el val_loss

El val_loss baja hasta hasta mantenerse entre 0.5 y 0.7, y el val baja hasta 0.75 y a partir de ahí se mantiene.

##### ¿Qué habeis hecho para evitar el overfitting y como os habeis asegurado de ello?



##### ¿Qué tal os va el modelo con el "test de la calle"?